In [ ]:
pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 42.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from google.colab import userdata, drive
from pathlib import Path
import gurobipy as gb

In [ ]:
drive.mount('/content/drive')
root = Path('/content/drive')
drv = root / 'MyDrive' / 'Group 13 - Project 1'

Mounted at /content/drive


In [ ]:
# Read in data and convert to pandas dataframes
income = pd.read_csv(drv / 'avg_individual_income.csv')
child_care_regulated = pd.read_csv(drv / "child_care_regulated.csv")
employment_rate = pd.read_csv(drv / "employment_rate.csv")
population_info = pd.read_csv(drv / "population.csv")
location_info = pd.read_csv(drv / "potential_locations.csv")

In [ ]:
# Rename zipcode column across different datasets so that we can merge datasets using this column
income = income.rename(columns={"ZIP code": "zipcode"})
child_care_regulated = child_care_regulated.rename(columns={"zip_code":"zipcode"})

In [ ]:
# Convert the zipcode column in all datasets to string
income["zipcode"] = income["zipcode"].astype(str)
child_care_regulated["zipcode"] = child_care_regulated["zipcode"].astype(str)
employment_rate["zipcode"] = employment_rate["zipcode"].astype(str)
population_info["zipcode"] = population_info["zipcode"].astype(str)
location_info["zipcode"] = location_info["zipcode"].astype(str)

In [ ]:
# Go through location_info dataset and process the rows where the zipcode is not of length 5
csv_list=[income,child_care_regulated,employment_rate,population_info,location_info]
for csv in csv_list:
    for i in range(len(csv)):
        zipcode_len = len(str(csv.loc[i,"zipcode"]))
        if zipcode_len == 3:
            csv.loc[i, "zipcode"] = "00" + str(csv.loc[i,"zipcode"]) # add 2 leading zeros if length of zipcode is 3
        elif zipcode_len == 4:
            csv.loc[i, "zipcode"] = "0" + str(csv.loc[i,"zipcode"]) # add 1 leading zero if length of zipcode is 4

In [ ]:
location_info.reset_index(inplace=True,drop=True)
income.reset_index(inplace=True,drop=True)
display(location_info)

,zipcode,latitude,longitude
0,00501,40.816376,-73.040796
1,00501,40.817455,-73.044202
2,00501,40.813873,-73.042182
3,00501,40.814152,-73.037265
4,00501,40.824673,-73.047431
...,...,...,...
215395,14905,42.077220,-76.849045
215396,14905,42.092659,-76.838969
215397,14905,42.091170,-76.830071
215398,14905,42.077221,-76.834010


In [ ]:
location_info[location_info["latitude"].isna()]

,zipcode,latitude,longitude


In [ ]:
# get list of all possible zipcodes from location_info dataset
summary = pd.DataFrame(location_info["zipcode"].unique().tolist(),columns=["zipcode"])
summary["zipcode"] = summary["zipcode"].astype(str)
display(summary)

,zipcode
0,00501
1,00544
2,06390
3,10001
4,10002
...,...
2149,14901
2150,14902
2151,14903
2152,14904


In [ ]:
# fix zipcodes in child_care_regulated dataset (some of the zipcodes were longer than length 5) so here we truncate them to just 5 digits

for i in range(len(child_care_regulated)):
    zip = str(child_care_regulated.loc[i, "zipcode"])
    if len(zip) > 5:
        new_zip = zip[:5]
        child_care_regulated.loc[i,"zipcode"] = int(new_zip)

# child_care_regulated.to_csv("updated child care info.csv",index=False)

In [ ]:
# There was some issue that was arising with the zipcodes where duplicates of a zipcode would show up in the dataframe.
# The below code strips any "invisible" spacing / news lines/ etc to make sure that all zipcodes are of the same format and to prevent any duplicates
current_center_zipcodes = pd.DataFrame(child_care_regulated["zipcode"].astype(str).str.strip().str.replace("\r", "").str.replace("\n", "").str.zfill(5).unique().tolist(),columns=["zipcode"])

# All of these zipcodes are from the child care regulated dataset and therefore imply that at least 1 child care center exists at this zipcode
current_center_zipcodes["Center Exists"] = "Y"

In [ ]:
current_center_zipcodes

,zipcode,Center Exists
0,13323,Y
1,14701,Y
2,12590,Y
3,10960,Y
4,12077,Y
...,...,...
1183,14536,Y
1184,12022,Y
1185,14058,Y
1186,14171,Y


In [ ]:
# merge these 2 dataframes so that we get a comprehensive list of all the zipcodes
summary = pd.merge(summary,current_center_zipcodes,on="zipcode",how="outer")

In [ ]:
# for all the zipcodes that don't have an existing center, fill value with "N"
summary["Center Exists"] = summary["Center Exists"].fillna("N")
display(summary)

,zipcode,Center Exists
0,00501,N
1,00544,N
2,06390,N
3,10001,Y
4,10002,Y
...,...,...
2153,14901,Y
2154,14902,N
2155,14903,Y
2156,14904,Y


In [ ]:
# merge employment rate and income data to summary dataframe
summary = pd.merge(summary,employment_rate, on="zipcode", how="left")
summary = pd.merge(summary,income, on="zipcode",how="left")
summary["zipcode"] = summary["zipcode"].astype(str)
display(summary)

,zipcode,Center Exists,employment rate,average income
0,00501,N,NaN,NaN
1,00544,N,NaN,NaN
2,06390,N,NaN,NaN
3,10001,Y,0.595097,102878.033603
4,10002,Y,0.520662,59604.041165
...,...,...,...,...
2153,14901,Y,NaN,44556.840077
2154,14902,N,NaN,NaN
2155,14903,Y,NaN,63550.420168
2156,14904,Y,NaN,44295.933735


In [ ]:
# HANDLING MISSING EMPLOYMENT RATE AND AVERAGE INCOME DATA FOR CERTAIN ZIPCODES

# Get first three digits of every zipcode
summary["zipcode group 1"] = summary["zipcode"].str.slice(0,3)
# Get mean of employment rate and average income of each zipcode group
avg_per_zipcode_group = summary[["zipcode group 1", "employment rate", "average income"]].groupby(["zipcode group 1"]).mean()
display(avg_per_zipcode_group)

,employment rate,average income
zipcode group 1,,
005,NaN,NaN
063,NaN,NaN
100,0.517505,94468.851717
101,0.525370,124664.006764
102,0.398990,143289.597068
103,0.471382,71256.024533
104,0.457156,47631.372837
105,0.511336,100811.248443
106,0.540386,86033.028691


In [ ]:
for i in range(len(summary)):
    zipcode = str(summary.loc[i,"zipcode"])
    zipcode_group = zipcode[:3]
    if pd.isna(summary.loc[i,"employment rate"]): # if the employment rate is missing for this zipcode
        summary.loc[i,"employment rate"] = avg_per_zipcode_group.loc[zipcode_group,"employment rate"] # replace NaN with corresponding mean employment rate of that zipcode group
    if pd.isna(summary.loc[i,"average income"]): # if the average income is missing for this zipcode
        summary.loc[i,"average income"] = avg_per_zipcode_group.loc[zipcode_group,"average income"] # replace NaN with corresponding mean income of that zipcode group

In [ ]:
summary

,zipcode,Center Exists,employment rate,average income,zipcode group 1
0,00501,N,NaN,NaN,005
1,00544,N,NaN,NaN,005
2,06390,N,NaN,NaN,063
3,10001,Y,0.595097,102878.033603,100
4,10002,Y,0.520662,59604.041165,100
...,...,...,...,...,...
2153,14901,Y,NaN,44556.840077,149
2154,14902,N,NaN,55184.608564,149
2155,14903,Y,NaN,63550.420168,149
2156,14904,Y,NaN,44295.933735,149


In [ ]:
# find the remaining zipcodes with missing data that should just be deleted
zipcodes_to_remove = summary[summary['average income'].isna() | summary["employment rate"].isna()]["zipcode"].tolist()
zipcodes_to_remove

['00501', '00544', '06390', '14901', '14902', '14903', '14904', '14905']

In [ ]:
# Remove these zipcodes from the summary dataframe
summary.dropna(inplace=True)
summary.reset_index(drop=True)

,zipcode,Center Exists,employment rate,average income,zipcode group 1
0,10001,Y,0.595097,102878.033603,100
1,10002,Y,0.520662,59604.041165,100
2,10003,Y,0.497244,114273.049645,100
3,10004,Y,0.506661,132004.310345,100
4,10005,Y,0.665833,121437.713311,100
...,...,...,...,...,...
2145,14893,N,0.565556,59311.089615,148
2146,14894,N,0.565556,54025.423729,148
2147,14895,Y,0.565556,54655.612245,148
2148,14897,Y,0.565556,54044.117647,148


In [ ]:
summary[summary["Center Exists"]=="N"]

,zipcode,Center Exists,employment rate,average income,zipcode group 1
10,10008,N,0.517505,94468.851717,100
19,10018,N,0.757593,108746.189024,100
42,10041,N,0.517505,94468.851717,100
43,10043,N,0.517505,94468.851717,100
45,10045,N,0.517505,94468.851717,100
...,...,...,...,...,...
2135,14878,N,0.565556,59926.470588,148
2144,14887,N,0.565556,59311.089615,148
2148,14893,N,0.565556,59311.089615,148
2149,14894,N,0.565556,54025.423729,148


In [ ]:
# Classify each zipcode area by high demand vs normal demand

# for high demand area, child care desert if # of available slots is <= 1/2 population of kids aged 2 wks to 12 yrs
# for normal demand area, child care desert if # of available slots <= 1/3 of population of kids

mask = (summary["employment rate"] >= 0.6) | (summary["average income"] <= 60000)
summary["Demand Type"] = np.where(mask, "High", "Normal")
summary

,zipcode,Center Exists,employment rate,average income,zipcode group 1,Demand Type
3,10001,Y,0.595097,102878.033603,100,Normal
4,10002,Y,0.520662,59604.041165,100,High
5,10003,Y,0.497244,114273.049645,100,Normal
6,10004,Y,0.506661,132004.310345,100,Normal
7,10005,Y,0.665833,121437.713311,100,High
...,...,...,...,...,...,...
2148,14893,N,0.565556,59311.089615,148,High
2149,14894,N,0.565556,54025.423729,148,High
2150,14895,Y,0.565556,54655.612245,148,High
2151,14897,Y,0.565556,54044.117647,148,High


In [ ]:
summary.reset_index(inplace=True,drop=True)

In [ ]:
summary.to_csv("Summary.csv",index=False) # writing to csv file just to double check the contents of the data for troubleshooting


In [ ]:
display(child_care_regulated)

,facility_id,program_type,facility_status,facility_name,city,zipcode,school_district_name,infant_capacity,toddler_capacity,preschool_capacity,school_age_capacity,children_capacity,total_capacity,latitude,longitude
0,2416,FDC,Registration,"Bohrer, Barbara",Clinton,13323,Clinton,0,0,0,2,6,8,NaN,NaN
1,5555,FDC,Registration,"Matey, Sally",Jamestown,14701,Jamestown,0,0,0,2,6,8,NaN,NaN
2,9066,FDC,Registration,"Copeland, Denise",Wappingers Falls,12590,Wappingers,0,0,0,2,6,8,NaN,NaN
3,40163,DCC,License,"Head Start of Rockland, Inc.",Nyack,10960,Nyack,0,10,110,0,0,120,41.089425,-73.920413
4,41016,SACC,Registration,"School's Out, Inc.",Glenmont,12077,Bethlehem,0,0,0,75,0,75,42.607043,-73.788606
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15599,892735,GFDC,License,LITTLE LILIES GROUP FAMILY DAYCARE LLC.,Bronx,10462,Bronx 11,0,0,0,4,12,16,40.854317,-73.864996
15600,897263,GFDC,License,"Cummings, Darlene",Brooklyn,11205,Brooklyn 13,0,0,0,0,10,10,40.696940,-73.977121
15601,901966,GFDC,License,"Pascal Genao, Angela",Yonkers,10705,Yonkers,0,0,0,4,12,16,40.910754,-73.893528
15602,892455,GFDC,License,"Warnakulasuriya, Sajeeka",Staten Island,10303,Richmond 31,0,0,0,4,12,16,40.628144,-74.156228


In [ ]:
# Compute under 5 capacity based on infant capacity, toddler capacity, and preschool capacity --> data assumption = we decided to take 60% of preschool capacity
child_care_regulated['under-5_capacity']=child_care_regulated["infant_capacity"]+child_care_regulated["toddler_capacity"]+0.6*child_care_regulated["preschool_capacity"]


In [ ]:
missing_location_centers = child_care_regulated[child_care_regulated["latitude"].isna() | child_care_regulated["longitude"].isna()]

In [ ]:
missing_location_zipcodes = missing_location_centers["zipcode"].unique().tolist()
num_zipcodes_missing = len(missing_location_zipcodes)
print(num_zipcodes_missing)

349


590 existing centers with missing latitude/longitude data \
349 zipcodes with missing latitude/longitude data --> delete altogether and do not use them in our model

In [ ]:
child_care_regulated[child_care_regulated["zipcode"]=="10308"]

,facility_id,program_type,facility_status,facility_name,city,zipcode,school_district_name,infant_capacity,toddler_capacity,preschool_capacity,school_age_capacity,children_capacity,total_capacity,latitude,longitude,under-5_capacity
3449,710051,GFDC,License,"Royzenshteyn, Anna",Staten Island,10308,Richmond 31,0,0,0,4,12,16,40.559380,-74.154682,0.0
3936,709759,SACC,Registration,"United Activities Unlimited, Inc. @ IS 24",Staten Island,10308,Richmond 31,0,0,0,144,0,144,40.545902,-74.145897,0.0
6398,275814,SACC,Registration,"United Activities Unlimited, Inc. @ P.S. 8",Staten Island,10308,Richmond 31,0,0,0,95,0,95,40.547820,-74.152115,0.0
6914,744199,GFDC,License,"Dream A Little Dream Daycare, LLC",Staten Island,10308,Richmond 31,0,0,0,4,12,16,40.552307,-74.158737,0.0
14310,909612,FDC,Registration,"Abdulhaq, Intisar",Staten island,10308,Richmond 31,0,0,0,2,6,8,40.539078,-74.143724,0.0
15188,891390,SACC,Registration,"United Activities Unlimited, Inc.",Staten Island,10308,Richmond 31,0,0,0,152,0,152,40.558924,-74.155164,0.0


In [ ]:
# get the total capacity across all existing childcare centers for each zipcode


child_care_regulated["zipcode"] = child_care_regulated["zipcode"].astype(str).str.strip().str.replace("\r","").str.replace("\n","").str.zfill(5)

capacity_by_zipcode = child_care_regulated[["zipcode","total_capacity",'under-5_capacity']].groupby(["zipcode"]).sum().reset_index()

display(capacity_by_zipcode)

,zipcode,total_capacity,under-5_capacity
0,10001,609,0.0
1,10002,4729,10.8
2,10003,1995,0.0
3,10004,263,0.0
4,10005,39,0.0
...,...,...,...
1183,14897,16,12.8
1184,14901,709,130.8
1185,14903,97,11.4
1186,14904,258,43.8


In [ ]:
capacity_by_zipcode[capacity_by_zipcode["zipcode"]=="10308"]

,zipcode,total_capacity,under-5_capacity
52,10308,447,0.0


In [ ]:
# Merge these two dataframes together
summary2 = pd.merge(summary,capacity_by_zipcode, on="zipcode",how="left")


In [ ]:
summary2

,zipcode,Center Exists,employment rate,average income,zipcode group 1,Demand Type,total_capacity,under-5_capacity
0,10001,Y,0.595097,102878.033603,100,Normal,609.0,0.0
1,10002,Y,0.520662,59604.041165,100,High,4729.0,10.8
2,10003,Y,0.497244,114273.049645,100,Normal,1995.0,0.0
3,10004,Y,0.506661,132004.310345,100,Normal,263.0,0.0
4,10005,Y,0.665833,121437.713311,100,High,39.0,0.0
...,...,...,...,...,...,...,...,...
2145,14893,N,0.565556,59311.089615,148,High,NaN,NaN
2146,14894,N,0.565556,54025.423729,148,High,NaN,NaN
2147,14895,Y,0.565556,54655.612245,148,High,601.0,206.0
2148,14897,Y,0.565556,54044.117647,148,High,16.0,12.8


In [ ]:
summary2 = summary2[~summary2["zipcode"].isin(missing_location_zipcodes)]
summary2

,zipcode,Center Exists,employment rate,average income,zipcode group 1,Demand Type,total_capacity,under-5_capacity
0,10001,Y,0.595097,102878.033603,100,Normal,609.0,0.0
2,10003,Y,0.497244,114273.049645,100,Normal,1995.0,0.0
3,10004,Y,0.506661,132004.310345,100,Normal,263.0,0.0
4,10005,Y,0.665833,121437.713311,100,High,39.0,0.0
5,10006,Y,0.631692,126377.118644,100,High,156.0,8.4
...,...,...,...,...,...,...,...,...
2145,14893,N,0.565556,59311.089615,148,High,NaN,NaN
2146,14894,N,0.565556,54025.423729,148,High,NaN,NaN
2147,14895,Y,0.565556,54655.612245,148,High,601.0,206.0
2148,14897,Y,0.565556,54044.117647,148,High,16.0,12.8


In [ ]:
summary2.loc[summary2["total_capacity"].isna()]
# NaN for total_capacity and under_5 capacity where center does not exist --> makes sense because we would need to build a center at this zipcode! total capacity and under 5 capacity are not needed here

,zipcode,Center Exists,employment rate,average income,zipcode group 1,Demand Type,total_capacity,under-5_capacity
7,10008,N,0.517505,94468.851717,100,Normal,NaN,NaN
16,10018,N,0.757593,108746.189024,100,High,NaN,NaN
39,10041,N,0.517505,94468.851717,100,Normal,NaN,NaN
40,10043,N,0.517505,94468.851717,100,Normal,NaN,NaN
42,10045,N,0.517505,94468.851717,100,Normal,NaN,NaN
...,...,...,...,...,...,...,...,...
2132,14878,N,0.565556,59926.470588,148,High,NaN,NaN
2141,14887,N,0.565556,59311.089615,148,High,NaN,NaN
2145,14893,N,0.565556,59311.089615,148,High,NaN,NaN
2146,14894,N,0.565556,54025.423729,148,High,NaN,NaN


966 zipcodes are zipcodes that do not have any existing childcare centers hence their total_capacity and under-5_capacity #s will be 0

In [ ]:
len(summary2) - len(summary2.loc[summary2["total_capacity"].isna()])

855

number of zipcodes that have existing childcare centers: 855

In [ ]:
summary2

,zipcode,Center Exists,employment rate,average income,zipcode group 1,Demand Type,total_capacity,under-5_capacity
0,10001,Y,0.595097,102878.033603,100,Normal,609.0,0.0
2,10003,Y,0.497244,114273.049645,100,Normal,1995.0,0.0
3,10004,Y,0.506661,132004.310345,100,Normal,263.0,0.0
4,10005,Y,0.665833,121437.713311,100,High,39.0,0.0
5,10006,Y,0.631692,126377.118644,100,High,156.0,8.4
...,...,...,...,...,...,...,...,...
2145,14893,N,0.565556,59311.089615,148,High,NaN,NaN
2146,14894,N,0.565556,54025.423729,148,High,NaN,NaN
2147,14895,Y,0.565556,54655.612245,148,High,601.0,206.0
2148,14897,Y,0.565556,54044.117647,148,High,16.0,12.8


In [ ]:
# replace the NaNs with 0s
summary2 = summary2.copy()
summary2["total_capacity"] = summary2["total_capacity"].fillna(0)
summary2["under-5_capacity"] = summary2["under-5_capacity"].fillna(0)
# if total capacity = 0, then no center exists at this zipcode

In [ ]:
summary2.to_csv("summary 2 check.csv")

In [ ]:
display(population_info)

,zipcode,Total,-5,5-9,10-14,15-19,20-24,25-29,30-34,35-39,40-44,45-49,50-54,55-59,60-64,65-69,70-74,75-79,80-84,85+
0,06390,53,0,1,5,0,6,0,9,18,0,12,2,0,0,0,0,0,0,0
1,10001,27004,744,784,942,1035,2296,3806,3588,2524,1702,1903,1704,1225,1323,933,815,616,488,576
2,10002,76518,2142,3046,3198,2652,4528,6988,6278,5157,4962,4822,4410,6106,4548,4815,4748,2531,2793,2794
3,10003,53877,1440,1034,953,7013,6344,7100,6427,3221,2907,1988,2698,2350,2274,2793,1854,1646,779,1056
4,10004,4579,433,182,161,108,109,601,724,490,241,313,549,279,199,173,2,15,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1641,14774,290,4,54,18,24,12,5,2,31,8,15,8,27,68,6,6,2,0,0
1642,14785,130,0,0,0,0,0,33,32,0,0,0,0,36,29,0,0,0,0,0
1643,14788,104,0,0,1,3,1,11,0,0,0,41,4,18,2,0,0,2,0,21
1644,14805,765,31,16,29,24,42,12,31,37,28,31,29,143,105,36,141,12,8,10


In [ ]:
# Take 100% of 0-5, 100% 5-9, and 60% of 10-14 as an approximation for the 10-12 year old age group --> ROUNDED TO NEAREST INTEGER
population_info.rename(columns={"-5":"0-4"},inplace=True) # Assume no overlap in the age groups 0-5 and 5-9. Convert 0-5 to 0-4 (inclusive) so that each age group is comprised of 5 ages
    # also matches what you see with other age groups ex) 20-25, 25-20, and so forth
population_info["10-12"] = population_info["10-14"] * 0.6
population_info["Total Kid Count"] = population_info["0-4"] + population_info["5-9"] + population_info["10-12"]

population_kids = population_info[["zipcode", "0-4", "5-9", "10-12", "Total Kid Count"]]
display(population_kids)

,zipcode,0-4,5-9,10-12,Total Kid Count
0,06390,0,1,3.0,4.0
1,10001,744,784,565.2,2093.2
2,10002,2142,3046,1918.8,7106.8
3,10003,1440,1034,571.8,3045.8
4,10004,433,182,96.6,711.6
...,...,...,...,...,...
1641,14774,4,54,10.8,68.8
1642,14785,0,0,0.0,0.0
1643,14788,0,0,0.6,0.6
1644,14805,31,16,17.4,64.4


In [ ]:
summary3 = pd.merge(summary2, population_kids, on="zipcode", how="left")
summary3

,zipcode,Center Exists,employment rate,average income,zipcode group 1,Demand Type,total_capacity,under-5_capacity,0-4,5-9,10-12,Total Kid Count
0,10001,Y,0.595097,102878.033603,100,Normal,609.0,0.0,744.0,784.0,565.2,2093.2
1,10003,Y,0.497244,114273.049645,100,Normal,1995.0,0.0,1440.0,1034.0,571.8,3045.8
2,10004,Y,0.506661,132004.310345,100,Normal,263.0,0.0,433.0,182.0,96.6,711.6
3,10005,Y,0.665833,121437.713311,100,High,39.0,0.0,484.0,204.0,137.4,825.4
4,10006,Y,0.631692,126377.118644,100,High,156.0,8.4,128.0,96.0,45.0,269.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1816,14893,N,0.565556,59311.089615,148,High,0.0,0.0,NaN,NaN,NaN,NaN
1817,14894,N,0.565556,54025.423729,148,High,0.0,0.0,NaN,NaN,NaN,NaN
1818,14895,Y,0.565556,54655.612245,148,High,601.0,206.0,NaN,NaN,NaN,NaN
1819,14897,Y,0.565556,54044.117647,148,High,16.0,12.8,NaN,NaN,NaN,NaN


In [ ]:

commercial_zipcodes = pd.read_csv(drv / "Data Cleaning/Commercial Zipcodes.csv")
commercial_zipcodes["zipcode"] = commercial_zipcodes["zipcode"].astype(str)

In [ ]:
summary3 = summary3[~summary3["zipcode"].isin(commercial_zipcodes["zipcode"])]
summary3

,zipcode,Center Exists,employment rate,average income,zipcode group 1,Demand Type,total_capacity,under-5_capacity,0-4,5-9,10-12,Total Kid Count
0,10001,Y,0.595097,102878.033603,100,Normal,609.0,0.0,744.0,784.0,565.2,2093.2
1,10003,Y,0.497244,114273.049645,100,Normal,1995.0,0.0,1440.0,1034.0,571.8,3045.8
2,10004,Y,0.506661,132004.310345,100,Normal,263.0,0.0,433.0,182.0,96.6,711.6
3,10005,Y,0.665833,121437.713311,100,High,39.0,0.0,484.0,204.0,137.4,825.4
4,10006,Y,0.631692,126377.118644,100,High,156.0,8.4,128.0,96.0,45.0,269.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1816,14893,N,0.565556,59311.089615,148,High,0.0,0.0,NaN,NaN,NaN,NaN
1817,14894,N,0.565556,54025.423729,148,High,0.0,0.0,NaN,NaN,NaN,NaN
1818,14895,Y,0.565556,54655.612245,148,High,601.0,206.0,NaN,NaN,NaN,NaN
1819,14897,Y,0.565556,54044.117647,148,High,16.0,12.8,NaN,NaN,NaN,NaN


In [ ]:
missing_summary = (
    summary3.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)
missing_summary

,column,missing_count
0,zipcode,0
1,Center Exists,0
2,employment rate,0
3,average income,0
4,zipcode group 1,0
5,Demand Type,0
6,total_capacity,0
7,under-5_capacity,0
8,0-4,458
9,5-9,458


In [ ]:
summary3.to_csv("summary dataframe with missing pop data.csv")

In [ ]:
df = summary3[summary3["0-4"].isna() | summary3["5-9"].isna() | summary3["10-12"].isna()]
missing_pop_zipcodes = df["zipcode"]
missing_pop_zipcodes.to_csv("zipcodes with missing population data.csv")

In [ ]:
public_population_data = pd.read_excel(drv / "Data Cleaning/Public Data - Population By Age Group.xls", sheet_name = "Population by Age Group")
display(public_population_data)

,ZipCode,Total Population,Age Group 0-4,Age Group 5-9,Age Group 10-14
0,6390,53,0,2,6
1,10001,29079,1173,889,907
2,10002,75517,2259,3268,3051
3,10003,53825,1399,999,926
4,10004,3875,364,284,73
...,...,...,...,...,...
1819,14898,1501,130,89,49
1820,14901,14430,582,953,1025
1821,14903,7140,430,331,323
1822,14904,14276,594,804,1029


In [ ]:
# Rename age group columns in the population dataset pulled from US Census Bureau
public_population_data = public_population_data.rename(columns={"ZipCode":"zipcode","Total Population":"Total","Age Group 0-4":"0-4","Age Group 5-9":"5-9","Age Group 10-14":"10-14"})

In [ ]:
public_population_data["zipcode"]= public_population_data["zipcode"].astype(str)
for i in range(len(public_population_data)):
    zipcode_len = len(str(public_population_data.loc[i,"zipcode"]))
    if zipcode_len == 4:
        public_population_data.loc[i,"zipcode"] = "0" + str(public_population_data.loc[i,"zipcode"])

display(public_population_data)


,zipcode,Total,0-4,5-9,10-14
0,06390,53,0,2,6
1,10001,29079,1173,889,907
2,10002,75517,2259,3268,3051
3,10003,53825,1399,999,926
4,10004,3875,364,284,73
...,...,...,...,...,...
1819,14898,1501,130,89,49
1820,14901,14430,582,953,1025
1821,14903,7140,430,331,323
1822,14904,14276,594,804,1029


In [ ]:
public_population_data["10-12"] = public_population_data["10-14"]*0.6
public_population_data["Total Kid Count"] = public_population_data["0-4"] + public_population_data["5-9"] + public_population_data["10-12"]

In [ ]:
public_population_data.set_index("zipcode", inplace=True)
public_population_data

,Total,0-4,5-9,10-14,10-12,Total Kid Count
zipcode,,,,,,
06390,53,0,2,6,3.6,5.6
10001,29079,1173,889,907,544.2,2606.2
10002,75517,2259,3268,3051,1830.6,7357.6
10003,53825,1399,999,926,555.6,2953.6
10004,3875,364,284,73,43.8,691.8
...,...,...,...,...,...,...
14898,1501,130,89,49,29.4,248.4
14901,14430,582,953,1025,615.0,2150.0
14903,7140,430,331,323,193.8,954.8


In [ ]:
df

,zipcode,Center Exists,employment rate,average income,zipcode group 1,Demand Type,total_capacity,under-5_capacity,0-4,5-9,10-12,Total Kid Count
6,10008,N,0.517505,94468.851717,100,Normal,0.0,0.0,NaN,NaN,NaN,NaN
33,10043,N,0.517505,94468.851717,100,Normal,0.0,0.0,NaN,NaN,NaN,NaN
35,10045,N,0.517505,94468.851717,100,Normal,0.0,0.0,NaN,NaN,NaN,NaN
37,10060,N,0.517505,94468.851717,100,Normal,0.0,0.0,NaN,NaN,NaN,NaN
41,10080,N,0.517505,94468.851717,100,Normal,0.0,0.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1816,14893,N,0.565556,59311.089615,148,High,0.0,0.0,NaN,NaN,NaN,NaN
1817,14894,N,0.565556,54025.423729,148,High,0.0,0.0,NaN,NaN,NaN,NaN
1818,14895,Y,0.565556,54655.612245,148,High,601.0,206.0,NaN,NaN,NaN,NaN
1819,14897,Y,0.565556,54044.117647,148,High,16.0,12.8,NaN,NaN,NaN,NaN


In [ ]:
count = 0
for idx in df.index:
    zipcode = df.loc[idx,"zipcode"]
    if zipcode in public_population_data.index:
        # print(zipcode)
        count += 1
print(count)

# 151 zipcodes can use the population data from US Census Bureau!

151


In [ ]:
summary3 = summary3.copy()
for idx in df.index:
    zipcode = df.loc[idx,"zipcode"]
    if (zipcode in public_population_data.index) & (zipcode == summary3.loc[idx,"zipcode"]) & (pd.isna(summary3.loc[idx,"Total Kid Count"])):
        # print("hi")
        # print(summary3.loc[idx,"0-4"])
        # print(public_population_data.loc[zipcode,"0-4"])
        summary3.loc[idx, "0-4"] = public_population_data.loc[zipcode,"0-4"]
        # print(summary3.loc[idx, "0-4"])
        summary3.loc[idx, "5-9"] = public_population_data.loc[zipcode,"5-9"]
        summary3.loc[idx, "10-12"] = public_population_data.loc[zipcode, "10-12"]
        summary3.loc[idx, "Total Kid Count"] = public_population_data.loc[zipcode, "Total Kid Count"]

In [ ]:
display(summary3)

,zipcode,Center Exists,employment rate,average income,zipcode group 1,Demand Type,total_capacity,under-5_capacity,0-4,5-9,10-12,Total Kid Count
0,10001,Y,0.595097,102878.033603,100,Normal,609.0,0.0,744.0,784.0,565.2,2093.2
1,10003,Y,0.497244,114273.049645,100,Normal,1995.0,0.0,1440.0,1034.0,571.8,3045.8
2,10004,Y,0.506661,132004.310345,100,Normal,263.0,0.0,433.0,182.0,96.6,711.6
3,10005,Y,0.665833,121437.713311,100,High,39.0,0.0,484.0,204.0,137.4,825.4
4,10006,Y,0.631692,126377.118644,100,High,156.0,8.4,128.0,96.0,45.0,269.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1816,14893,N,0.565556,59311.089615,148,High,0.0,0.0,0.0,0.0,10.8,10.8
1817,14894,N,0.565556,54025.423729,148,High,0.0,0.0,85.0,104.0,19.8,208.8
1818,14895,Y,0.565556,54655.612245,148,High,601.0,206.0,496.0,308.0,361.8,1165.8
1819,14897,Y,0.565556,54044.117647,148,High,16.0,12.8,51.0,80.0,33.0,164.0


In [ ]:
missing_summary = (
    summary3.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)
missing_summary

,column,missing_count
0,zipcode,0
1,Center Exists,0
2,employment rate,0
3,average income,0
4,zipcode group 1,0
5,Demand Type,0
6,total_capacity,0
7,under-5_capacity,0
8,0-4,307
9,5-9,307


Just delete these 307 zipcodes

In [ ]:
summary3 = summary3.dropna().copy()

In [ ]:
summary3

,zipcode,Center Exists,employment rate,average income,zipcode group 1,Demand Type,total_capacity,under-5_capacity,0-4,5-9,10-12,Total Kid Count
0,10001,Y,0.595097,102878.033603,100,Normal,609.0,0.0,744.0,784.0,565.2,2093.2
1,10003,Y,0.497244,114273.049645,100,Normal,1995.0,0.0,1440.0,1034.0,571.8,3045.8
2,10004,Y,0.506661,132004.310345,100,Normal,263.0,0.0,433.0,182.0,96.6,711.6
3,10005,Y,0.665833,121437.713311,100,High,39.0,0.0,484.0,204.0,137.4,825.4
4,10006,Y,0.631692,126377.118644,100,High,156.0,8.4,128.0,96.0,45.0,269.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1816,14893,N,0.565556,59311.089615,148,High,0.0,0.0,0.0,0.0,10.8,10.8
1817,14894,N,0.565556,54025.423729,148,High,0.0,0.0,85.0,104.0,19.8,208.8
1818,14895,Y,0.565556,54655.612245,148,High,601.0,206.0,496.0,308.0,361.8,1165.8
1819,14897,Y,0.565556,54044.117647,148,High,16.0,12.8,51.0,80.0,33.0,164.0


In [ ]:
# for high demand area, child care desert if # of available slots is <= 1/2 population of kids aged 2 wks to 12 yrs
# for normal demand area, child care desert if # of available slots <= 1/3 of population of kids

high = summary3["Demand Type"] == "High"
norm = summary3["Demand Type"] == "Normal"
mask = high & (summary3["total_capacity"] <= (1/2) * summary3["Total Kid Count"])
summary3["Desert"] = np.where(mask, "Y", np.where(norm & (summary3["total_capacity"] <= (1/3) * summary3["Total Kid Count"]),"Y", "N"))

In [ ]:
# get only the zipcodes that are determined to be childcare deserts
desert_zipcodes = summary3[summary3["Desert"]=="Y"].reset_index(drop=True)

In [ ]:
display(desert_zipcodes)

,zipcode,Center Exists,employment rate,average income,zipcode group 1,Demand Type,total_capacity,under-5_capacity,0-4,5-9,10-12,Total Kid Count,Desert
0,10001,Y,0.595097,102878.033603,100,Normal,609.0,0.0,744.0,784.0,565.2,2093.2,Y
1,10005,Y,0.665833,121437.713311,100,High,39.0,0.0,484.0,204.0,137.4,825.4,Y
2,10007,Y,0.528910,138853.904282,100,Normal,284.0,0.0,605.0,451.0,174.0,1230.0,Y
3,10010,Y,0.492749,116272.698810,100,Normal,234.0,0.0,1422.0,1592.0,568.8,3582.8,Y
4,10012,Y,0.538273,111131.786340,100,Normal,24.0,0.0,613.0,161.0,244.2,1018.2,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1308,14892,Y,0.565556,55246.478873,148,High,52.0,9.6,237.0,373.0,255.6,865.6,Y
1309,14893,N,0.565556,59311.089615,148,High,0.0,0.0,0.0,0.0,10.8,10.8,Y
1310,14894,N,0.565556,54025.423729,148,High,0.0,0.0,85.0,104.0,19.8,208.8,Y
1311,14897,Y,0.565556,54044.117647,148,High,16.0,12.8,51.0,80.0,33.0,164.0,Y


In [ ]:
desert_zipcodes.to_csv("Childcare Deserts.csv")